**Wojciech Olejnik**

# Apriori

_Celem laboratorium było wykonanie implementacji algorytmu Apriori._

In [7]:
from csv import reader
from collections import defaultdict
from itertools import chain, combinations

In [23]:
# Wczytanie danych z pliku CSV

def get_data_from_csv(fname):
    itemSets = []
    itemSet = set()

    with open(fname, 'r') as file:
        csv_reader = reader(file)
        for line in csv_reader:
            line = list(filter(None, line))          
            record = set(line)
            for item in record:
                itemSet.add(frozenset([item]))
            itemSets.append(record)
    return itemSet, itemSets

# Wybieramy produktów, których wsparcie jest większe od zadanego

def get_above_min_support(itemSet, itemSetList, minSup, globalItemSetWithSup):
    freqItemSet = set()
    localItemSetWithSup = defaultdict(int)

    for item in itemSet:
        for itemSet in itemSetList:
            if item.issubset(itemSet):
                globalItemSetWithSup[item] += 1
                localItemSetWithSup[item] += 1
    for item, supCount in localItemSetWithSup.items():
        support = float(supCount / len(itemSetList))
        if(support >= minSup):
            freqItemSet.add(item)
            print(f'{item} ==> {support}')

    return freqItemSet

# Gernerowanie kandydatów

def generate_candiadates(candidateSet, prevFreqSet, length):
    
# Aby wykonać testowanie podzbiorów, wykonujemy pętlę przez wszystkie możliwe podzbiory w zbiorze elementów.
# Jeśli dany podzbiór nie znajduje się w poprzednim częstym zbiorze, usuwamy go.
    tempCandidateSet = candidateSet.copy()
    for item in candidateSet:
        subsets = combinations(item, length)
        for subset in subsets:
            if(frozenset(subset) not in prevFreqSet):
                tempCandidateSet.remove(item);
                break

    return tempCandidateSet

# Tworzenie reguł

def associationRule(freqItemSet, itemSetWithSup, minConf):
    rules = []
    for k, itemSet in freqItemSet.items():
        for item in itemSet:
            subsets = chain.from_iterable(combinations(item, r) for r in range(1, len(item)))
            for s in subsets:
                confidence = float(
                    itemSetWithSup[item] / itemSetWithSup[frozenset(s)])
                if(confidence > minConf):
                    rules.append([set(s), set(item.difference(s)), confidence])
    return rules

# główny program

def apriori(fname, minSup, minConf):
    itemSet, itemSetList = get_data_from_csv(fname)

    globalFreqItemSet = dict() # czestotliwość produktu
    globalItemSetWithSup = defaultdict(int) # czestotliwość zestawów z wsparciem

    itemCurrentSetTmp = get_above_min_support(
        itemSet, itemSetList, minSup, globalItemSetWithSup)
    currentLSet = itemCurrentSetTmp
    k = 2

    while(currentLSet): # obliczanie częstotliwość zbiorów

        globalFreqItemSet[k-1] = currentLSet
        # wszystkie połączenia poprzez brute-force i zwracamy tylko te, które są w określonej długości.
        candidateSet = set()
        for i in currentLSet:
            for j in currentLSet:
                if len(i.union(j)) == k:
                    candidateSet.add(i.union(j))

        candidateSet = generate_candiadates(candidateSet, currentLSet, k-1)
        currentLSet = get_above_min_support( candidateSet, itemSetList, minSup, globalItemSetWithSup)
        k += 1

    rules = associationRule(globalFreqItemSet, globalItemSetWithSup, minConf)
    rules.sort(key=lambda x: x[2])

    return globalFreqItemSet, rules


**Zadanie 1**

Przetesować działanie na małym zbiorze w postaci transakcji.
Wygenrowany zbiór

In [22]:
itemSet, itemSets = get_data_from_csv('test_data.csv')

['B', 'A']
['B', 'D', 'C']
['D', 'A', 'C', 'E']
['D', 'A', 'E']
['B', 'A', 'C']
['B', 'A', 'D', 'C']
['B', 'C']
['B', 'A', 'C']
['B', 'A', 'D']
['B', 'E', 'C']


In [36]:
print('Pattern ==> support')
freqItemSet, rules = apriori('test_data.csv', 0.05, 0.1)

print('\n------------------------------------\n')
print('if product_1 then product_2 ==> confidence')
for rule in rules:
    print(f'If {rule[0] } then {rule[1]} ==> {rule[2]}')

Pattern ==> support
frozenset({'B'}) ==> 0.8
frozenset({'D'}) ==> 0.5
frozenset({'E'}) ==> 0.3
frozenset({'A'}) ==> 0.7
frozenset({'C'}) ==> 0.7
frozenset({'B', 'D'}) ==> 0.3
frozenset({'A', 'D'}) ==> 0.4
frozenset({'E', 'D'}) ==> 0.2
frozenset({'A', 'E'}) ==> 0.2
frozenset({'E', 'C'}) ==> 0.2
frozenset({'A', 'B'}) ==> 0.5
frozenset({'B', 'E'}) ==> 0.1
frozenset({'B', 'C'}) ==> 0.6
frozenset({'A', 'C'}) ==> 0.4
frozenset({'D', 'C'}) ==> 0.3
frozenset({'A', 'E', 'D'}) ==> 0.2
frozenset({'B', 'D', 'C'}) ==> 0.2
frozenset({'B', 'E', 'C'}) ==> 0.1
frozenset({'A', 'B', 'D'}) ==> 0.2
frozenset({'A', 'D', 'C'}) ==> 0.2
frozenset({'E', 'D', 'C'}) ==> 0.1
frozenset({'A', 'E', 'C'}) ==> 0.1
frozenset({'A', 'B', 'C'}) ==> 0.3
frozenset({'A', 'E', 'C', 'D'}) ==> 0.1
frozenset({'A', 'C', 'B', 'D'}) ==> 0.1

------------------------------------

if product_1 then product_2 ==> confidence
If {'B'} then {'E'} ==> 0.125
If {'B'} then {'E', 'C'} ==> 0.125
If {'B'} then {'A', 'D', 'C'} ==> 0.125
If {'C'}

**Zadanie 2**

Przetesować działanie na zbiorze store_data.csv.

In [35]:
print('Pattern ==> support')
freqItemSet, rules = apriori('store_data.csv', 0.05, 0.1)

print('\n------------------------------------\n')
print('if product_1 then product_2 ==> confidence')
for rule in rules:
    print(f'If {rule[0] } then {rule[1]} ==> {rule[2]}')

Pattern ==> support
frozenset({'green tea'}) ==> 0.13211571790427942
frozenset({'milk'}) ==> 0.12958272230369283
frozenset({'spaghetti'}) ==> 0.17411011865084655
frozenset({'chocolate'}) ==> 0.1638448206905746
frozenset({'grated cheese'}) ==> 0.0523930142647647
frozenset({'cake'}) ==> 0.08105585921877083
frozenset({'frozen smoothie'}) ==> 0.06332489001466471
frozenset({'burgers'}) ==> 0.0871883748833489
frozenset({'pancakes'}) ==> 0.09505399280095987
frozenset({'soup'}) ==> 0.05052659645380616
frozenset({'mineral water'}) ==> 0.23836821757099053
frozenset({'ground beef'}) ==> 0.09825356619117451
frozenset({'olive oil'}) ==> 0.0658578856152513
frozenset({'cooking oil'}) ==> 0.0510598586855086
frozenset({'tomatoes'}) ==> 0.06839088121583789
frozenset({'chicken'}) ==> 0.05999200106652446
frozenset({'whole wheat rice'}) ==> 0.058525529929342755
frozenset({'eggs'}) ==> 0.17970937208372217
frozenset({'french fries'}) ==> 0.1709105452606319
frozenset({'escalope'}) ==> 0.0793227569657379
froze